# Real $Cl(4,4)$, split octonions, and triality

This generated SolveIt-style notebook checks the exact real algebra and both pure-Rust CVODE studies. It treats historical notebook output as evidence only and uses the repository's independent checkers.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
assert (ROOT / 'Cargo.toml').is_file()
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
def run_check(script, *arguments):
    process = subprocess.run(
        [sys.executable, str(ROOT / script), *arguments],
        cwd=ROOT, text=True, capture_output=True, check=False
    )
    print(process.stdout.strip())
    assert process.returncode == 0, process.stderr
    return process.stdout
EXPECTED_HASHES = {'cl44': '0660e436fdcf90ed0f0481c9a828e92659987f5820084ccbf5e1dfc96866cc54', 'split': '25044ea194c1e18e40341da1f0098b80bfeb9f4bcb6e7665a32d03f55ac360ca', 'triality': '5985f5c0fdd6656c3c9f572ef1a5d06eee21aabe44cc9bf33747bb79b5e181dd', 'transport': '61b8fe5a8a72e48d40d40d6850a6c559e2dd06ccfa9de97982590d35a82044fb', 'cosmology': '8f1e45a37c14cc10c7c6b6469ceca7c65e7b9ffc08b3c9dd5bfc518065c590db'}


## Independent exact-algebra checks

The checks below use standard-library integer and rational arithmetic; they do not trust stored Wolfram results.

In [ ]:
run_check('scripts/check_cl44_fixture.py')
run_check('scripts/check_split_octonion_fixture.py')
run_check('scripts/check_triality44_fixture.py')


## CVODE studies

The transport study evolves 24 real states. The cosmology study evolves 18 real states and is compared against analytic background identities.

In [ ]:
run_check('scripts/check_triality_transport.py')
run_check('scripts/check_spinor_cosmology.py')
transport = json.loads((ROOT / 'artifacts/triality-transport/summary.json').read_text())
cosmology = json.loads((ROOT / 'artifacts/spinor-cosmology/summary.json').read_text())
print({'transport_samples': transport['sampleCount'], 'cosmology_samples': cosmology['sampleCount']})


## Dataset inspection

The notebook reads the canonical CSV files directly and checks their endpoint grids and row counts.

In [ ]:
def read_csv(relative_path):
    with (ROOT / relative_path).open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))
transport_rows = read_csv('artifacts/triality-transport/trajectory.csv')
cosmology_rows = read_csv('artifacts/spinor-cosmology/background.csv')
assert len(transport_rows) == 41
assert len(cosmology_rows) == 1201
assert float(transport_rows[0]['t']) == 0.0
assert float(transport_rows[-1]['t']) == 4.0
assert float(cosmology_rows[0]['N']) == -4.0
assert float(cosmology_rows[-1]['N']) == 1.0
{'transport_final': transport_rows[-1], 'cosmology_present': cosmology_rows[960]}


## Reproducibility report

In [ ]:
actual_hashes = {
    'cl44': sha256(ROOT / 'artifacts/exact/cl44-seed.json'),
    'split': sha256(ROOT / 'artifacts/exact/split-octonion.json'),
    'triality': sha256(ROOT / 'artifacts/exact/triality44.json'),
    'transport': sha256(ROOT / 'artifacts/triality-transport/summary.json'),
    'cosmology': sha256(ROOT / 'artifacts/spinor-cosmology/summary.json'),
}
NOTEBOOK_CHECKS = {
    'input_hashes': actual_hashes == EXPECTED_HASHES,
    'transport_success': transport['verdict'] == 'SUCCESS',
    'transport_dimension': transport['stateDimension'] == 24,
    'cosmology_success': cosmology['verdict'] == 'SUCCESS',
    'cosmology_dimension': cosmology['stateDimension'] == 18,
    'dataset_counts': len(transport_rows) == 41 and len(cosmology_rows) == 1201,
}
assert all(NOTEBOOK_CHECKS.values()), NOTEBOOK_CHECKS
report_path = ROOT / 'artifacts/notebooks/jupyter-report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report = {'schemaVersion': 1, 'checks': NOTEBOOK_CHECKS, 'inputSha256': actual_hashes}
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
NOTEBOOK_CHECKS
